# Compound IC50 Workflow

This notebook validates and optionally runs a ChEMBL IC50 query-composition workflow.

The descriptor keeps `execution.chembl_pages_to_fetch: 1` so live ChEMBL retrieval stays small. Live execution requires internet access.

In [2]:
import json
import subprocess
import sys
from pathlib import Path
from pprint import pprint

from bioseq_dl.cli.workflows import load_workflow_recipe, validate_workflow_recipe

repo_root = Path.cwd().parent.parent
config_path = repo_root / "examples" / "workflows" / "compound_chembl_ic50_ranges.yml"
output_dir = repo_root / "examples" / "results" / "compound_chembl_ic50_ranges"

recipe = load_workflow_recipe(config_path)
normalized = validate_workflow_recipe(recipe)

pprint({
    "query": normalized["query"],
    "mode": normalized["mode"],
    "chembl_pages_to_fetch": normalized["chembl_pages_to_fetch"],
})

{'chembl_pages_to_fetch': 1,
 'mode': 'query_composition',
 'query': 'ic50:0-10=very_high_potency,ic50:10-100=high_potency'}


## Optional live run

Set `run_live_workflow = True` to call ChEMBL when internet access is available.

In [5]:
run_live_workflow = True

if run_live_workflow:
    command = [
        sys.executable,
        "-m",
        "bioseq_dl.cli.main",
        "workflow",
        "run",
        "--config",
        str(config_path),
    ]
    result = subprocess.run(command, cwd=repo_root, capture_output=True, text=True, check=False)
    print(result.stdout)
    if result.returncode != 0:
        print("Live execution failed; internet access or ChEMBL availability may be missing.")
        print(result.stderr)
else:
    print("Live workflow execution skipped.")

2026-06-18 15:57:40 | INFO     | bioseq_dl.core.workflow.main | Pipeline: fetching ChEMBL for query=standard_type=IC50 AND standard_value>0 AND standard_value<10 search_type=activity
2026-06-18 15:57:40 | INFO     | bioseq_dl.interfaces.chembl | Fetching page: https://www.ebi.ac.uk/chembl/api/data/activity.json?standard_type=IC50&format=json&limit=100&standard_value__gt=0&standard_value__lt=10 for method activity-search with pages_to_fetch=1
2026-06-18 15:57:47 | INFO     | bioseq_dl.core.workflow.main | Pipeline: fetching UniProt for query=(xref:chembl-CHEMBL357 OR xref:chembl-CHEMBL613075 OR xref:chembl-CHEMBL369 OR xref:chembl-CHEMBL612558 OR xref:chembl-CHEMBL1944 OR xref:chembl-CHEMBL285 OR xref:chembl-CHEMBL1902 OR xref:chembl-CHEMBL1907609 OR xref:chembl-CHEMBL3849 OR xref:chembl-CHEMBL389 OR xref:chembl-CHEMBL3332 OR xref:chembl-CHEMBL2093863 OR xref:chembl-CHEMBL2061 OR xref:chembl-CHEMBL1855 OR xref:chembl-CHEMBL2094250 OR xref:chembl-CHEMBL2871 OR xref:chembl-CHEMBL2094125 O

## Inspect metadata and labels

If outputs already exist, the metadata document can be inspected without another API call.

In [6]:
metadata_path = output_dir / "metadata.json"

if metadata_path.exists():
    metadata = json.loads(metadata_path.read_text(encoding="utf-8"))
    normalized_descriptor = metadata.get("normalized_descriptor", {})
    query_descriptor = normalized_descriptor.get("query", {})
    pprint({
        "schema_version": normalized_descriptor.get("schema_version"),
        "composition": query_descriptor.get("composition"),
        "output_files": metadata.get("output_files", []),
    })
else:
    print("No metadata.json exists yet. Run the workflow when internet access is available.")

{'composition': [{'description': 'IC50 values from 0 to 10 in the workflow '
                                 'query syntax.',
                  'label': 'very_high_potency',
                  'value': 'ic50:0-10'},
                 {'description': 'IC50 values from 10 to 100 in the workflow '
                                 'query syntax.',
                  'label': 'high_potency',
                  'value': 'ic50:10-100'}],
 'output_files': [{'category': 'result',
                   'column_names': ['_id',
                                    'action_type',
                                    'activity_comment',
                                    'activity_id',
                                    'activity_properties',
                                    'assay_chembl_id',
                                    'assay_description',
                                    'assay_type',
                                    'assay_variant_accession',
                                    'assay